# 🧬 Módulo 4: Modelado de Proteínas y Docking Molecular
## Actividad 4.4: Preparación de Proteínas para Docking

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_04_modelado_proteinas_docking/04_preparacion_proteinas.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Limpiar y preparar estructuras de proteínas
- Adicionar hidrógenos correctamente
- Asignar cargas atómicas apropiadas
- Optimizar orientación de residuos
- Tratar aguas cristalográficas
- Usar PDBFixer, PyMOL y otras herramientas

---

In [ ]:
# Instalación de dependencias
!pip install biopython requests py3Dmol numpy pandas matplotlib
# pdbfixer requiere OpenMM
!pip install pdbfixer || conda install -c conda-forge pdbfixer -y 2>/dev/null || echo "Instala pdbfixer con: conda install -c conda-forge pdbfixer"

In [ ]:
import requests
import py3Dmol
from Bio import PDB
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import PPBuilder
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ Bibliotecas importadas correctamente")

## 📚 Introducción

La preparación adecuada de una estructura proteica es un paso **crítico** antes de cualquier estudio de docking o dinámica molecular. Una preparación incorrecta puede generar resultados erróneos o poco reproducibles.

### Problemas Comunes en Estructuras Cristalográficas

| Problema | Causa | Solución |
|----------|-------|---------|
| Residuos faltantes | Regiones desordenadas en el cristal | Modelado de loops |
| Átomos de hidrógeno ausentes | No visibles en rayos X | Adición computacional |
| Moléculas de agua | Agua cristalográfica | Selección/eliminación |
| Heteroátomos (ligandos, iones) | Co-cristalización | Eliminar o conservar |
| Cadenas múltiples | Unidad asimétrica | Seleccionar monómero |
| Residuos con ocupancia parcial | Conformaciones alternativas | Elegir conformación A |
| Histidinas protonadas | Ambigüedad en pH fisiológico | Asignar estado de protonación |

### Flujo de Trabajo

```
PDB/AlphaFold → Inspección → Limpieza → PDBFixer → Hidrógenos → Cargas → Archivo de salida
```

## 1. Descarga e Inspección de la Estructura

In [ ]:
def descargar_pdb(pdb_id, output_dir="estructuras_preparacion"):
    """Descarga un archivo PDB desde RCSB."""
    Path(output_dir).mkdir(exist_ok=True)
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    response = requests.get(url)
    if response.status_code == 200:
        out_path = Path(output_dir) / f"{pdb_id}.pdb"
        out_path.write_text(response.text)
        print(f"✓ {pdb_id}.pdb descargado ({len(response.text)//1024} KB)")
        return out_path
    else:
        print(f"✗ Error descargando {pdb_id}")
        return None

def inspeccionar_pdb(pdb_file):
    """Inspección detallada de un archivo PDB."""
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prot", pdb_file)
    
    resumen = {
        "cadenas": [],
        "heteroatomos": [],
        "aguas": 0,
        "residuos_faltantes": []
    }
    
    for model in structure:
        for chain in model:
            residuos_std = [r for r in chain if r.id[0] == ' ']
            heteroat = [r for r in chain if r.id[0].startswith('H_')]
            aguas = [r for r in chain if r.id[0] == 'W']
            
            resumen["cadenas"].append({
                "id": chain.id,
                "residuos": len(residuos_std),
                "heteroatomos": len(heteroat),
                "aguas": len(aguas)
            })
            resumen["aguas"] += len(aguas)
            for h in heteroat:
                resumen["heteroatomos"].append(f"{h.resname} ({chain.id}:{h.id[1]})")
    
    print(f"\n{'='*50}")
    print(f"  INSPECCIÓN: {pdb_file}")
    print(f"{'='*50}")
    for c in resumen["cadenas"]:
        print(f"  Cadena {c['id']}: {c['residuos']} residuos, "
              f"{c['heteroatomos']} heteroátomos, {c['aguas']} aguas")
    if resumen["heteroatomos"]:
        print(f"\n  Heteroátomos encontrados:")
        for h in set(resumen["heteroatomos"]):
            print(f"    • {h}")
    print(f"\n  Total moléculas de agua: {resumen['aguas']}")
    print(f"{'='*50}\n")
    return structure

# Usaremos la proteína COX-2 (1CX2) - target clásico de fármacos AINEs
pdb_id = "1CX2"
pdb_file = descargar_pdb(pdb_id)
if pdb_file:
    estructura = inspeccionar_pdb(pdb_file)

## 2. Limpieza de la Estructura

### Qué conservar y qué eliminar

- **Conservar**: Cadena proteica principal, iones metálicos funcionales, ligando de interés
- **Eliminar**: Moléculas de agua (generalmente), ligandos no relevantes, cadenas simétricas, átomos de hidrógeno (se re-agregarán)

In [ ]:
class SelectorProteina(Select):
    """Selector para conservar sólo la cadena proteica principal sin agua."""
    
    def __init__(self, cadena='A', eliminar_agua=True, eliminar_hetero=True):
        self.cadena = cadena
        self.eliminar_agua = eliminar_agua
        self.eliminar_hetero = eliminar_hetero
    
    def accept_chain(self, chain):
        return chain.id == self.cadena
    
    def accept_residue(self, residue):
        # Eliminar agua
        if self.eliminar_agua and residue.id[0] == 'W':
            return False
        # Eliminar heteroátomos (ligandos, etc.)
        if self.eliminar_hetero and residue.id[0].startswith('H_'):
            return False
        return True

def limpiar_estructura(pdb_file, cadena='A', output_dir="estructuras_preparacion"):
    """Limpia la estructura PDB conservando sólo la cadena proteica."""
    parser = PDBParser(QUIET=True)
    estructura = parser.get_structure("prot", pdb_file)
    
    output_file = Path(output_dir) / f"{Path(pdb_file).stem}_limpio.pdb"
    
    io = PDBIO()
    io.set_structure(estructura)
    io.save(str(output_file), SelectorProteina(cadena=cadena))
    
    # Contar residuos en archivo limpio
    nueva_struct = parser.get_structure("limpio", output_file)
    n_residuos = sum(1 for chain in nueva_struct[0] 
                     for res in chain if res.id[0] == ' ')
    
    print(f"✓ Estructura limpiada guardada en: {output_file}")
    print(f"  Residuos proteicos: {n_residuos}")
    return output_file

# Limpiar la estructura de COX-2
pdb_limpio = limpiar_estructura(pdb_file, cadena='A')

# Verificar la limpieza
print("\n--- VERIFICACIÓN ---")
_ = inspeccionar_pdb(pdb_limpio)

## 3. Corrección de Residuos Faltantes con PDBFixer

`PDBFixer` es una herramienta de OpenMM que puede:
- Completar residuos que faltan en la secuencia
- Agregar átomos faltantes en residuos incompletos
- Modelar loops faltantes
- Reemplazar residuos no estándar

In [ ]:
def reparar_con_pdbfixer(pdb_file, output_dir="estructuras_preparacion", ph=7.4):
    """
    Repara una estructura PDB usando PDBFixer.
    - Completa residuos y átomos faltantes
    - Agrega hidrógenos al pH indicado
    
    Nota: requiere 'pdbfixer' instalado (conda install -c conda-forge pdbfixer)
    """
    try:
        from pdbfixer import PDBFixer
        from openmm.app import PDBFile
        import openmm as mm
        
        fixer = PDBFixer(filename=str(pdb_file))
        
        # 1. Encontrar cadenas/residuos faltantes
        fixer.findMissingResidues()
        fixer.findNonstandardResidues()
        
        print(f"Residuos no estándar: {fixer.nonstandardResidues}")
        fixer.replaceNonstandardResidues()
        
        # 2. Eliminar heteroátomos (opcional: conservarlos si son el ligando de interés)
        fixer.removeHeterogens(keepWater=False)
        
        # 3. Agregar átomos faltantes (incluyendo loops)
        fixer.findMissingAtoms()
        fixer.addMissingAtoms()
        
        # 4. Agregar hidrógenos al pH fisiológico
        fixer.addMissingHydrogens(ph)
        
        # Guardar resultado
        output_file = Path(output_dir) / f"{Path(pdb_file).stem}_preparado.pdb"
        with open(output_file, 'w') as f:
            PDBFile.writeFile(fixer.topology, fixer.positions, f)
        
        print(f"✓ Estructura reparada guardada en: {output_file}")
        print(f"  pH usado para hidrógenos: {ph}")
        return output_file
        
    except ImportError:
        print("⚠️  PDBFixer no está instalado.")
        print("   Instálalo con: conda install -c conda-forge pdbfixer")
        print("\n   Alternativa: usando BioPython para reparación básica...")
        return reparar_basico_biopython(pdb_file, output_dir)

def reparar_basico_biopython(pdb_file, output_dir="estructuras_preparacion"):
    """Reparación básica con BioPython (sin hidrógenos)."""
    parser = PDBParser(QUIET=True)
    estructura = parser.get_structure("prot", pdb_file)
    
    output_file = Path(output_dir) / f"{Path(pdb_file).stem}_reparado_bio.pdb"
    io = PDBIO()
    io.set_structure(estructura)
    io.save(str(output_file))
    
    print(f"✓ Estructura guardada (básica): {output_file}")
    return output_file

# Aplicar PDBFixer a la estructura limpia
pdb_preparado = reparar_con_pdbfixer(pdb_limpio, ph=7.4)

## 4. Análisis de Residuos Problemáticos

### Histidinas: el caso más crítico

Las histidinas (HIS) pueden estar protonadas en:
- **HID**: H en delta-N (N$\delta$)
- **HIE**: H en epsilon-N (N$\varepsilon$) - más común
- **HIP**: doblemente protonado (catiónico, carga +1)

La asignación incorrecta puede alterar significativamente las interacciones en el sitio activo.

In [ ]:
def analizar_residuos_criticos(pdb_file):
    """Identifica y reporta residuos que requieren atención especial."""
    parser = PDBParser(QUIET=True)
    estructura = parser.get_structure("prot", pdb_file)
    
    resumen = {
        "histidinas": [],
        "cisteinas": [],
        "metioninas": [],
        "protonacion_sugerida": []
    }
    
    residuos_especiales = {
        'HIS': 'Histidina (verificar estado de protonación)',
        'CYS': 'Cisteína (verificar puentes disulfuro)',
        'MET': 'Metionina (susceptible a oxidación)',
        'GLU': 'Glutamato',
        'ASP': 'Aspartato'
    }
    
    for model in estructura:
        for chain in model:
            for residue in chain:
                res_name = residue.resname.strip()
                if res_name == 'HIS':
                    resumen["histidinas"].append(f"HIS {chain.id}:{residue.id[1]}")
                elif res_name == 'CYS':
                    resumen["cisteinas"].append(f"CYS {chain.id}:{residue.id[1]}")
    
    print("🔍 ANÁLISIS DE RESIDUOS CRÍTICOS")
    print("="*45)
    print(f"\nHistidinas encontradas: {len(resumen['histidinas'])}")
    for h in resumen['histidinas'][:10]:  # Mostrar hasta 10
        print(f"  • {h} → Verificar HID/HIE/HIP")
    
    print(f"\nCisteínas encontradas: {len(resumen['cisteinas'])}")
    for c in resumen['cisteinas'][:5]:
        print(f"  • {c} → Verificar puentes S-S")
    
    print("\n💡 Recomendación: Use PROPKA o H++ para asignación automática")
    print("   PROPKA: https://www.ddl.unimi.it/~propka/")
    print("   H++:    http://newbiophysics.cs.vt.edu/H++/")
    
    return resumen

resultado = analizar_residuos_criticos(pdb_limpio)

## 5. Verificación de Puentes Disulfuro

Los puentes disulfuro (S-S) entre cisteínas son cruciales para la estabilidad estructural de muchas proteínas. Se identifican por la distancia entre átomos SG (< 2.1 Å) de dos cisteínas.

In [ ]:
def detectar_puentes_disulfuro(pdb_file, umbral=2.5):
    """
    Detecta posibles puentes disulfuro midiendo distancias SG-SG.
    
    Args:
        pdb_file: archivo PDB
        umbral: distancia máxima (Å) para considerar puente S-S (valor real ~2.05 Å)
    """
    parser = PDBParser(QUIET=True)
    estructura = parser.get_structure("prot", pdb_file)
    
    # Recolectar todos los átomos SG de cisteínas
    atomos_sg = []
    for model in estructura:
        for chain in model:
            for residue in chain:
                if residue.resname == 'CYS':
                    if 'SG' in residue:
                        atomos_sg.append({
                            'residuo': residue,
                            'cadena': chain.id,
                            'numero': residue.id[1],
                            'sg': residue['SG']
                        })
    
    puentes = []
    print(f"Cisteínas con SG encontradas: {len(atomos_sg)}")
    
    for i in range(len(atomos_sg)):
        for j in range(i+1, len(atomos_sg)):
            sg1 = atomos_sg[i]['sg']
            sg2 = atomos_sg[j]['sg']
            distancia = sg1 - sg2  # BioPython usa __sub__ para distancia
            
            if distancia < umbral:
                puente = (f"CYS {atomos_sg[i]['cadena']}:{atomos_sg[i]['numero']} "
                         f"-- CYS {atomos_sg[j]['cadena']}:{atomos_sg[j]['numero']}")
                puentes.append((puente, distancia))
                print(f"  🔗 Puente S-S: {puente}  d={distancia:.2f} Å")
    
    if not puentes:
        print("  No se detectaron puentes disulfuro.")
    else:
        print(f"\n  Total puentes disulfuro: {len(puentes)}")
    
    return puentes

puentes = detectar_puentes_disulfuro(pdb_limpio)

## 6. Análisis de Factores B (B-factors)

Los factores B (también llamados factores de temperatura) indican la movilidad o desorden atómico en la estructura cristalográfica:
- **B < 20 Å²**: Región rígida y bien definida
- **B 20-50 Å²**: Movilidad moderada
- **B > 50 Å²**: Alta movilidad o desorden — mayor incertidumbre en las coordenadas

In [ ]:
def analizar_factores_b(pdb_file):
    """Analiza y grafica los factores B de la estructura."""
    parser = PDBParser(QUIET=True)
    estructura = parser.get_structure("prot", pdb_file)
    
    b_values = []
    residue_ids = []
    
    for model in estructura:
        for chain in model:
            for residue in chain:
                if residue.id[0] == ' ':  # Sólo aminoácidos estándar
                    ca_b = None
                    if 'CA' in residue:
                        ca_b = residue['CA'].bfactor
                    if ca_b is not None:
                        b_values.append(ca_b)
                        residue_ids.append(residue.id[1])
    
    if not b_values:
        print("No se encontraron valores B.")
        return
    
    b_arr = np.array(b_values)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # Perfil a lo largo de la secuencia
    ax1.plot(residue_ids, b_arr, linewidth=1.5, color='steelblue')
    ax1.axhline(50, color='red', linestyle='--', alpha=0.7, label='B=50 (alta movilidad)')
    ax1.axhline(20, color='orange', linestyle='--', alpha=0.7, label='B=20 (límite rígido)')
    ax1.set_xlabel('Número de residuo')
    ax1.set_ylabel('Factor B (Å²)')
    ax1.set_title('Perfil de Factores B')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # Histograma
    ax2.hist(b_arr, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax2.axvline(np.mean(b_arr), color='red', linestyle='--', 
                label=f'Media: {np.mean(b_arr):.1f} Å²')
    ax2.set_xlabel('Factor B (Å²)')
    ax2.set_ylabel('Frecuencia')
    ax2.set_title('Distribución de Factores B')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle(f'Análisis de Factores B — {Path(pdb_file).stem}', 
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\nEstadísticas de Factores B:")
    print(f"  Media: {np.mean(b_arr):.2f} Å²")
    print(f"  Mín:   {np.min(b_arr):.2f} Å²")
    print(f"  Máx:   {np.max(b_arr):.2f} Å²")
    print(f"  Residuos con B > 50: {np.sum(b_arr > 50)} ({100*np.mean(b_arr > 50):.1f}%)")

analizar_factores_b(pdb_limpio)

## 7. Visualización con Py3Dmol

In [ ]:
def visualizar_estructura_preparada(pdb_file, titulo="Proteína Preparada"):
    """Visualiza la proteína preparada con Py3Dmol."""
    with open(pdb_file, 'r') as f:
        pdb_data = f.read()
    
    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_data, 'pdb')
    
    # Estilo cartoon con coloreado por cadena
    view.setStyle({'cartoon': {'colorscheme': 'chain'}})
    
    # Resaltar sitio activo si se conoce (para COX-2, residuos clave)
    # Arginina 120, Tirosina 355, Serina 530 son residuos del sitio activo de COX-2
    for res_id in [120, 355, 530]:
        view.addStyle(
            {'resi': res_id, 'chain': 'A'},
            {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.3}}
        )
    
    view.zoomTo()
    view.setBackgroundColor('white')
    
    print(f"Visualizando: {titulo}")
    print("Naranja: residuos del sitio activo (COX-2)")
    return view

view = visualizar_estructura_preparada(pdb_limpio, "COX-2 Preparada (1CX2)")
view.show()

## 8. Ejercicios Prácticos

### Ejercicio 1 (Básico)
Descarga la estructura de la proteína quinasa CDK2 (PDB: **1HCL**), realiza la limpieza básica seleccionando sólo la cadena A, y reporta cuántos residuos, moléculas de agua y heteroátomos tiene.

### Ejercicio 2 (Intermedio)
Para la proteína HIV-1 proteasa (PDB: **1HTF**):
1. Descarga y limpia la estructura
2. Detecta los puentes disulfuro
3. Analiza los factores B — ¿qué regiones son más móviles?

### Ejercicio 3 (Avanzado)
Compara la estructura de la lisozima sin preparar (PDB: **1AKI**) con la misma estructura después de usar PDBFixer:
1. ¿Cuántos residuos/átomos se añadieron?
2. Visualiza ambas estructuras con Py3Dmol
3. Documenta las diferencias en un resumen

In [ ]:
# Espacio para tus ejercicios
# Ejercicio 1: CDK2 (1HCL)
pdb_cdk2 = descargar_pdb("1HCL")
# Tu código aquí...

## 9. Referencias

1. Case, D.A. et al. (2021). *Amber 2021*. University of California, San Francisco.
2. Eastman, P. et al. (2017). OpenMM 7: Rapid development of high performance algorithms for molecular dynamics. *PLOS Comput. Biol.*
3. Hamelberg, D. & McCammon, J.A. (2009). Fast Peptidyl cis-trans Isomerization within the Flexible Gly-Rich Flaps of HIV-1 Protease. *J. Am. Chem. Soc.*
4. Word, J.M. et al. (1999). Asparagine and glutamine: using hydrogen atom contacts in the choice of side-chain amide orientation. *J. Mol. Biol.*
5. Olsson, M.H.M. et al. (2011). PROPKA3: Consistent Treatment of Internal and Surface Residues in Empirical pKa Predictions. *J. Chem. Theory Comput.*

---

## 📚 Recursos Adicionales

### Herramientas
- [PDBFixer](https://github.com/openmm/pdbfixer) — Corrección automática de estructuras PDB
- [PROPKA](https://www.ddl.unimi.it/~propka/) — Predicción de pKa y estado de protonación
- [H++](http://newbiophysics.cs.vt.edu/H++/) — Servidor web para adición de hidrógenos
- [PyMOL](https://pymol.org/) — Visualización y preparación avanzada
- [UCSF Chimera/ChimeraX](https://www.cgl.ucsf.edu/chimerax/) — Análisis y preparación
- [MolProbity](http://molprobity.biochem.duke.edu/) — Validación de geometría proteica

### Bases de Datos
- [RCSB PDB](https://www.rcsb.org/) — Protein Data Bank
- [PDBe](https://www.ebi.ac.uk/pdbe/) — Acceso europeo al PDB

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Descargar y explorar estructuras PDB con BioPython
- ✅ Limpiar estructuras eliminando agua y heteroátomos
- ✅ Identificar residuos problemáticos (HIS, CYS)
- ✅ Usar PDBFixer para completar residuos faltantes
- ✅ Detectar puentes disulfuro por análisis de distancias
- ✅ Interpretar factores B para evaluar movilidad estructural

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 4.4: Preparación de Proteínas para Docking**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_4.3-AlphaFold_y_ESMFold-blue.svg)](03_alphafold_esmfold.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_4.5_➡️-Fundamentos_de_Docking-green.svg)](05_docking_fundamentos.ipynb)

---

📚 **[Volver al Módulo 4](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>